# Grounding Full Benchmark - Adopted SOTA Serving Path

**Author**: Konrad Jelen<br>
**Approach**: end-to-end benchmark of the round-2 adopted grounder (stage-0 cosine gate → early-exit reranker → cascade band → NLI + logistic stack) over the full 2,752-claim gold on the deployed CPU OpenVINO int8 engines:

1. Fit the deployed calibration - logistic full-fit on the round-1 pair cache, thresholds from the established OOF protocol
2. Precompute unique-chunk vectors (warm RAG-cache assumption) and compile LATENCY-hint engines
3. Run the real serving path per claim, recording verdict, stage path and wall-clock latency
4. Score the full metric set - macro-F1, accuracy, per-class precision / recall / F1, confusion matrix - plus the latency distribution

Positive class for the flagging metrics is **hallucination** (gold label 0); a claim counts as flagged when the grounder rejects it. Project convention throughout: FP = false flags (supported claims rejected), FN = missed hallucinations.

## GPU Selection

CPU-only run - the deployed target is CPU int8, so CUDA is hidden before any framework import to keep every engine on the OpenVINO CPU plugin.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""        # CPU-only - deployed target is CPU int8
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # quiet tokenizer fork warnings

## Imports

Serving primitives from `grounding_openvino`, the calibration helpers from the hypotheses driver and ensemble module, and sklearn metrics for the evaluation.

In [ ]:
%load_ext autoreload
%autoreload 2

# Standard library
import time

# Data processing
import numpy as np

# Plotting
import matplotlib.pyplot as plt

# Metrics and calibration
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Project
import grounding_hypotheses as gh
from grounding_ensemble import best_macro
from grounding_models import load_gold
from grounding_openvino import (
    CASCADE_BAND,
    COSINE_GATE,
    _async_embed,
    compile_ir,
    embed_vectors,
    load_ov_hf,
    nli_max,
    rerank_max_early,
)

# Progress
from tqdm.auto import tqdm

# Rich console output
from rich import print as rprint
from rich.table import Table

## Configuration

The adopted serving operating point - everything below is the deployed configuration from rounds 1 and 2, nothing is swept here.

In [ ]:
K = 8                    # top-k pre-filtered chunks fed to the cross-encoders
SCHEDULE = (1, 1, 2, 4)  # early-exit progressive batch schedule (H13)
BAND = CASCADE_BAND      # adopted H11 cascade band (reranker-only outside, NLI inside)
GATE = COSINE_GATE       # adopted H12 stage-0 cosine gate (flag edge, pass edge)

rprint(f"""[white]Configuration[/white]
  Top-k: [dark_sea_green]{K}[/dark_sea_green]   exit schedule: [dark_sea_green]{SCHEDULE}[/dark_sea_green]
  Cascade band: [dark_sea_green]{BAND}[/dark_sea_green]   cosine gate: [dark_sea_green]{GATE}[/dark_sea_green]
  Engines: [cyan]bge-m3 / bge-reranker-v2-m3 / mDeBERTa-v3-nli[/cyan] [dim](OpenVINO int8, LATENCY hint)[/dim]
""")

## Data Loading

The full conversation-grounding gold - every claim with its retrieved evidence chunks and the human supported / hallucinated label.

In [ ]:
recs = load_gold()
y = np.array([r["label"] for r in recs])
hall = y == 0  # gold convention: label 0 = hallucination

rprint(f"gold: [dark_sea_green]{len(recs)}[/dark_sea_green] claims - "
       f"[dark_sea_green]{hall.sum()}[/dark_sea_green] hallucinated ([dark_sea_green]{hall.mean():.1%}[/dark_sea_green]), "
       f"[dark_sea_green]{(~hall).sum()}[/dark_sea_green] supported")

## Deployed Calibration

Logistic stack fit once on the round-1 pair cache features (rr_max, nli_ent_max), with the reranker-only threshold T_rr and stack threshold T_st taken from the established OOF best-macro protocol. Applied frozen to the live serving scores below.

In [ ]:
feats, y_full, _ = gh.build_features()
X_cal = np.column_stack([feats["rr_max"], feats["nli_ent_max"]])
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
clf.fit(X_cal, y_full)
_, T_rr, _ = best_macro(feats["rr_max"], y_full)
_, T_st, _ = best_macro(gh.oof_p(X_cal, y_full), y_full)

rprint(f"deployed calibration: T_rr=[dark_sea_green]{T_rr:.3f}[/dark_sea_green]  "
       f"T_st=[dark_sea_green]{T_st:.3f}[/dark_sea_green]")

## Engines and Chunk Precompute

LATENCY-hint engines for the per-claim serving loop, plus the warm-regime precompute: every unique evidence chunk embedded once on a THROUGHPUT engine (the RAG-cache assumption - chunk vectors exist before the claim arrives, so the loop only ever embeds the claim).

In [ ]:
_, etok, edir = load_ov_hf("bge-m3", compile=False)
_, rtok, rdir = load_ov_hf("bge-reranker-v2-m3", compile=False)
_, ntok, ndir = load_ov_hf("mDeBERTa-v3-nli", compile=False)

emb = compile_ir(edir / "openvino_model.xml", hint="LATENCY")
rr = compile_ir(rdir / "openvino_model.xml", hint="LATENCY")
nli = compile_ir(ndir / "openvino_model.xml", hint="LATENCY")

In [ ]:
emb_tp = compile_ir(edir / "openvino_model.xml", hint="THROUGHPUT")
uchunk: dict[str, int] = {}
for r in recs:
    for c in r["chunks"]:
        uchunk.setdefault(c, len(uchunk))
chunk_texts = list(uchunk)
order = np.argsort([len(t) for t in chunk_texts])
v = _async_embed(emb_tp, etok, [chunk_texts[i] for i in order], desc="embed chunks")
chvec_u = np.empty_like(v)
chvec_u[order] = v
del emb_tp

## Execution - Serving Loop

The real per-claim serving path, timed end to end: claim embed → cosine rank → stage-0 gate → early-exit reranker over the top-k → cascade band → NLI + logistic stack for in-band claims. Engines are warmed on three claims first so cold-compile noise stays out of the distribution.

In [ ]:
for r in recs[:3]:  # engine warm-up
    embed_vectors(emb, etok, [r["claim"]])
    rerank_max_early(rr, rtok, r["claim"], r["chunks"][:2], pass_edge=np.inf,
                     schedule=SCHEDULE)
    nli_max(nli, ntok, r["claim"], r["chunks"][:2])

In [ ]:
lat, category, flag = [], [], []
for r in tqdm(recs, desc="serving"):
    dv = chvec_u[[uchunk[c] for c in r["chunks"]]]
    t0 = time.perf_counter()
    cv = embed_vectors(emb, etok, [r["claim"]])[0]
    cos = dv @ cv
    cmax = float(cos.max())
    if cmax <= GATE[0]:
        cat, fl = "gate-flag", True
    elif cmax >= GATE[1]:
        cat, fl = "gate-pass", False
    else:
        ranked = [r["chunks"][j] for j in np.argsort(-cos)[:K]]
        s, _ = rerank_max_early(rr, rtok, r["claim"], ranked,
                                pass_edge=BAND[1], schedule=SCHEDULE)
        if s >= BAND[1]:
            cat, fl = "rr-pass", False
        elif s <= BAND[0]:
            cat, fl = "rr-flag", bool(s < T_rr)
        else:
            e = nli_max(nli, ntok, r["claim"], ranked)
            p = clf.predict_proba([[s, e]])[0, 1]
            cat, fl = "in-band", bool(p < T_st)
    lat.append(time.perf_counter() - t0)
    category.append(cat)
    flag.append(fl)

flag = np.array(flag)
category = np.array(category)
lat_ms = np.array(lat) * 1000.0

## Evaluation - Quality Metrics

Full metric set with hallucination as the positive class: confusion matrix, accuracy, macro-F1, and per-class precision / recall / F1 - as tables, then as a heatmap and per-class bar chart. The off-diagonal cells are the two project error types - false flags (supported claims rejected) and missed hallucinations.

In [ ]:
t = hall.astype(int)  # 1 = hallucination (positive class)
p = flag.astype(int)  # 1 = flagged by the grounder

cm = confusion_matrix(t, p, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()  # fp = false flags, fn = missed hallucinations

acc = accuracy_score(t, p)
macro = f1_score(t, p, average="macro")
p_h, r_h, f_h, _ = precision_recall_fscore_support(t, p, average="binary", pos_label=1)
p_s, r_s, f_s, _ = precision_recall_fscore_support(t, p, average="binary", pos_label=0)

cmt = Table(title=f"Confusion matrix (n={len(y)})")
cmt.add_column("")
cmt.add_column("predicted pass", justify="right")
cmt.add_column("predicted flag", justify="right")
cmt.add_row("actual supported", f"[dark_sea_green]{tn}[/dark_sea_green]",
            f"[indian_red]{fp}[/indian_red] [dim](FP - false flags)[/dim]")
cmt.add_row("actual hallucination", f"[indian_red]{fn}[/indian_red] [dim](FN - missed)[/dim]",
            f"[dark_sea_green]{tp}[/dark_sea_green]")

mt = Table(title="Quality metrics (positive class = hallucination)")
mt.add_column("metric")
mt.add_column("value", justify="right")
mt.add_row("macro-F1", f"[dark_sea_green]{macro:.3f}[/dark_sea_green]")
mt.add_row("accuracy", f"[dark_sea_green]{acc:.3f}[/dark_sea_green]")
mt.add_row("precision (hallucination)", f"[dark_sea_green]{p_h:.3f}[/dark_sea_green]")
mt.add_row("recall (hallucination)", f"[dark_sea_green]{r_h:.3f}[/dark_sea_green]")
mt.add_row("F1 (hallucination)", f"[dark_sea_green]{f_h:.3f}[/dark_sea_green]")
mt.add_row("precision (supported)", f"[dark_sea_green]{p_s:.3f}[/dark_sea_green]")
mt.add_row("recall (supported)", f"[dark_sea_green]{r_s:.3f}[/dark_sea_green]")
mt.add_row("F1 (supported)", f"[dark_sea_green]{f_s:.3f}[/dark_sea_green]")

rprint(cmt)
rprint(mt)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# confusion matrix heatmap - off-diagonal cells are the two project error types
ax = axes[0]
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], ["pass", "flag"])
ax.set_yticks([0, 1], ["supported", "hallucination"])
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title(f"Confusion matrix (n={len(y)})")
notes = {(0, 1): "FP - false flags", (1, 0): "FN - missed"}
for (i, j), v in np.ndenumerate(cm):
    label = f"{v:,}" + (f"\n{notes[(i, j)]}" if (i, j) in notes else "")
    ax.text(j, i, label, ha="center", va="center", fontsize=11,
            color="white" if v > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax, shrink=0.8)

# per-class precision / recall / F1 against the macro-F1 line
ax = axes[1]
x = np.arange(3)
w = 0.36
ax.bar(x - w / 2, [p_h, r_h, f_h], w, label="hallucination", color="indianred")
ax.bar(x + w / 2, [p_s, r_s, f_s], w, label="supported", color="darkseagreen")
ax.axhline(macro, ls="--", lw=1, color="gray", label=f"macro-F1 {macro:.3f}")
for xi, v in zip(x - w / 2, [p_h, r_h, f_h]):
    ax.text(xi, v + 0.015, f"{v:.3f}", ha="center", fontsize=9)
for xi, v in zip(x + w / 2, [p_s, r_s, f_s]):
    ax.text(xi, v + 0.015, f"{v:.3f}", ha="center", fontsize=9)
ax.set_xticks(x, ["precision", "recall", "F1"])
ax.set_ylim(0, 1.08)
ax.set_title("Per-class metrics")
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

## Evaluation - Latency

Warm per-claim wall-clock distribution over the full gold, plus the stage-path composition - which exit each claim took and what that path costs. Rendered as tables, then as a distribution histogram with percentile markers and a per-path boxplot.

In [ ]:
paths = ("gate-flag", "gate-pass", "rr-flag", "rr-pass", "in-band")
lt = Table(title="Stage-path composition and cost")
lt.add_column("path")
lt.add_column("share", justify="right")
lt.add_column("mean ms", justify="right")
lt.add_column("median ms", justify="right")
for c in paths:
    m = category == c
    if m.any():
        lt.add_row(c, f"{m.mean():.1%}",
                   f"[dark_sea_green]{lat_ms[m].mean():.0f}[/dark_sea_green]",
                   f"[dark_sea_green]{np.median(lat_ms[m]):.0f}[/dark_sea_green]")

rprint(lt)
rprint(f"""[white]Latency (n={len(recs)}, k={K}, LATENCY hint, warm)[/white]
  mean: [dark_sea_green]{lat_ms.mean():.0f} ms[/dark_sea_green]   median: [dark_sea_green]{np.median(lat_ms):.0f} ms[/dark_sea_green]   p90: [dark_sea_green]{np.percentile(lat_ms, 90):.0f} ms[/dark_sea_green]   p99: [dark_sea_green]{np.percentile(lat_ms, 99):.0f} ms[/dark_sea_green]
""")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# warm latency distribution with percentile markers
ax = axes[0]
ax.hist(lat_ms, bins=80, color="steelblue", alpha=0.85)
for q, name in ((50, "median"), (90, "p90"), (99, "p99")):
    v = np.percentile(lat_ms, q)
    ax.axvline(v, ls="--", lw=1, color="indianred")
    ax.text(v, ax.get_ylim()[1] * 0.97, f" {name} {v:.0f} ms", rotation=90,
            va="top", fontsize=9, color="indianred")
ax.set_xlabel("per-claim latency (ms)")
ax.set_ylabel("claims")
ax.set_title(f"Warm latency distribution (mean {lat_ms.mean():.0f} ms)")

# per-stage-path latency - the spend-where-uncertain shape of the cascade
ax = axes[1]
present = [c for c in paths if (category == c).any()]
data = [lat_ms[category == c] for c in present]
labels = [f"{c}\n{(category == c).mean():.1%}" for c in present]
bp = ax.boxplot(data, vert=False, tick_labels=labels, showfliers=False,
                patch_artist=True)
for b in bp["boxes"]:
    b.set_facecolor("darkseagreen")
    b.set_alpha(0.7)
ax.invert_yaxis()
ax.set_xlabel("per-claim latency (ms)")
ax.set_title("Latency by stage path (share of claims)")

plt.tight_layout()
plt.show()

## Summary

The full-gold end-to-end run reproduces the earlier script run exactly - identical verdicts and stage composition (the serving path is deterministic); only wall-clock latency moves with host load.

- **Quality**: macro-F1 **0.789**, accuracy 0.818; hallucination class precision 0.652 / recall 0.781 / F1 0.711; supported class precision 0.905 / recall 0.833 / F1 0.868
- **Confusion matrix**: 1,638 correct passes, 614 caught hallucinations, **328 false flags (FP)**, **172 missed hallucinations (FN)**
- **Composition**: 22% of claims resolve at the stage-0 gate, 41.4% at the reranker (early exit), 36.5% pay the full in-band path
- **Latency (this run)**: mean 492 ms / median 238 ms / p90 1,103 ms / p99 1,924 ms - vs 585 / 258 / 1,342 / 1,592 ms for the script run, at identical verdicts; absolute ms are CPU-load-sensitive, quality is the regression-stable signal

The error mix sits recall-heavy vs the OOF simulation (FP 328 / FN 172 vs 245 / 216) because serving maxes are over the top-8 pre-filtered chunks while the calibration used all-chunks maxes - re-fit thresholds on serving-derived scores before fixing a deployment operating point.